In [ ]:
import Shadow
import numpy as np
from ophyd import Signal, SignalRO, Component, Device
from optlnls.plot import plot_beam
from optlnls.importing import read_shadow_beam
from bluesky import RunEngine
from bluesky.plans import scan
from bluesky.callbacks.best_effort import BestEffortCallback

RE = RunEngine()
bec = BestEffortCallback()
RE.subscribe(bec)

In [ ]:
class ShadowEnergy(Signal):

    def set(self, value: float, **kwargs):
        sts = super().set(value, **kwargs)
        self.parent.oe.PH1 = value - self.parent.delta_energy / 2
        self.parent.oe.PH2 = value + self.parent.delta_energy / 2
        return sts

class ShadowAttribute(Signal):

    def set(self, value: float, **kwargs):
        sts = super().set(value, **kwargs)
        oe = self.parent.set_standard_parameters(self.parent.energy.get())
        self.parent.oe = oe
        return sts


class ShadowSource(Device):

    energy = Component(ShadowEnergy, kind="hinted")
    sigmax = Component(ShadowAttribute, kind="hinted")

    def __init__(
        self, name: str, beam: SignalRO, energy: float, delta_energy: float, n_rays: int = 1_000_000
    ):
        super().__init__(name=name)
        self.beam = beam.get()
        self.delta_energy = delta_energy
        self.n_rays = n_rays
        self.oe = self.set_standard_parameters(energy)
        self.energy.set(energy).wait()
        self.sigmax.set(0.033).wait()

    def set_standard_parameters(self, energy):
        oe = Shadow.Source()
        oe.SIGDIX = 0.01
        oe.SIGDIZ = 0.00113
        oe.SIGMAZ = 0.027
        oe.SIGMAX = self.sigmax.get()
        oe.VDIV1 = 0.001
        oe.VDIV2 = 0.001
        oe.FDISTR = 3
        oe.F_COLOR = 3
        oe.F_PHOT = 0
        oe.HDIV1 = 0.004
        oe.HDIV2 = 0.004
        oe.IDO_VX = 0
        oe.IDO_VZ = 0
        oe.IDO_X_S = 0
        oe.IDO_Y_S = 0
        oe.IDO_Z_S = 0
        oe.ISTAR1 = 5676561
        oe.NPOINT = self.n_rays
        oe.PH1 = energy - self.delta_energy / 2
        oe.PH2 = energy + self.delta_energy / 2
        return oe

    def read(self, **kwargs):
        self.beam.genSource(self.oe)
        # beam2D = read_shadow_beam(
        #         beam=self.beam,
        #         x_column_index=3,
        #         y_column_index=1
        # )
        # plot_beam(
        #     beam2D=beam2D,
        #     show_plot=True,
        #     fitType=3,
        #     textA=2,textB=5,
        #     x_range=1,x_range_min=-0.1,x_range_max=0.1,
        #     y_range=1,y_range_min=-0.1,y_range_max=0.1,
        #     zero_pad_x=2
        # )
        return super().read(**kwargs)


In [ ]:
class ShadowM1(Device):
    def __init__(self, beam: Shadow.Beam, past_element: list, **kwargs):
        super().__init__(**kwargs)
        self.beam = beam.get()
        self.past_element = past_element
        self.idx = 0
        oe = Shadow.OE()
        oe.ALPHA = 90.0
        oe.CIL_ANG = 90.0
        oe.DUMMY = 0.1
        oe.FCYL = 1
        oe.FHIT_C = 1
        oe.FMIRR = 1
        oe.FWRITE = 1
        oe.F_EXT = 1
        oe.RLEN1 = 180.0
        oe.RLEN2 = 180.0
        oe.RMIRR = 2108.8753452152337
        oe.RWIDX1 = 20.0
        oe.RWIDX2 = 20.0
        oe.T_IMAGE = 35.0
        oe.T_INCIDENCE = 80.825
        oe.T_REFLECTION = 80.825
        oe.T_SOURCE = 0.0
        self.oe = oe

    def read(self, **kwargs):
        self.past_element.read()
        self.beam.traceOE(self.oe, self.idx)
        beam2D = read_shadow_beam(
                beam=self.beam,
                x_column_index=3,
                y_column_index=1
        )
        plot_beam(
            beam2D=beam2D,
            show_plot=True,
            fitType=3,
            textA=2,textB=5,
            x_range=1,x_range_min=-0.2,x_range_max=0.2,
            y_range=1,y_range_min=-0.2,y_range_max=0.2,
            zero_pad_x=2
        )
        return super().read(**kwargs)

In [ ]:
class ShadowPitch(Signal):
    def set(self, value: float, **kwargs):
        sts = super().set(value, **kwargs)
        print(value)
        oe = Shadow.OE()
        oe.ALPHA = 180.0
        oe.DUMMY = 0.1
        oe.FCYL = 1
        oe.FHIT_C = 1
        oe.FMIRR = 1
        oe.FWRITE = 1
        oe.F_DEFAULT = 0
        oe.F_MOVE = 1
        oe.RLEN1 = 300.0
        oe.RLEN2 = 300.0
        oe.RWIDX1 = 20.0
        oe.RWIDX2 = 20.0
        oe.SIMAG = 14095.0
        oe.SSOUR = 14095.0
        oe.THETA = 82.75
        oe.T_IMAGE = 19.05
        oe.T_INCIDENCE = 82.75
        oe.T_REFLECTION = 82.75
        oe.T_SOURCE = 0.0
        oe.X_ROT = np.degrees(value)
        self.parent.oe = oe
        return sts

class ShadowM2(Device):
    pitch = Component(ShadowPitch)

    def __init__(self, beam: Shadow.Beam, past_element: list, pitch: float, **kwargs):
        super().__init__(**kwargs)
        self.idx = 1
        self.beam = beam.get()
        self.past_element = past_element
        oe = Shadow.OE()
        oe.ALPHA = 180.0
        oe.DUMMY = 0.1
        oe.FCYL = 1
        oe.FHIT_C = 1
        oe.FMIRR = 1
        oe.FWRITE = 1
        oe.F_DEFAULT = 0
        oe.F_MOVE = 1
        oe.RLEN1 = 300.0
        oe.RLEN2 = 300.0
        oe.RWIDX1 = 20.0
        oe.RWIDX2 = 20.0
        oe.SIMAG = 14095.0
        oe.SSOUR = 14095.0
        oe.THETA = 82.75
        oe.T_IMAGE = 19.05
        oe.T_INCIDENCE = 82.75
        oe.T_REFLECTION = 82.75
        oe.T_SOURCE = 0.0
        self.oe = oe
        self.pitch.set(pitch).wait()

    def read(self, **kwargs):
        self.past_element.read()
        self.beam.traceOE(self.oe, self.idx)
        beam2D = read_shadow_beam(
                beam=self.beam,
                x_column_index=3,
                y_column_index=1
        )
        plot_beam(
            beam2D=beam2D,
            show_plot=True,
            fitType=3,
            textA=2,textB=5,
            x_range=1,x_range_min=-0.3,x_range_max=0.2,
            y_range=1,y_range_min=-0.3,y_range_max=0.2,
            zero_pad_x=2
        )
        return super().read(**kwargs)

In [ ]:
class OphyBeam(Device):

    width = Component(SignalRO, kind="hinted")

    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.beam = Shadow.Beam()

    def get(self, **kwargs):
        return self.beam

In [ ]:
beamline = []
Shadow.Beam()

beam = OphyBeam(name="beam")

beamline.append(
    ShadowSource(
        name="src",
        beam = beam,
        energy=20000,
        delta_energy=1000,
        n_rays=100_000
))

beamline.append(
    ShadowM1(
        name="m1",
        beam = beam,
        past_element=beamline[-1]
))

beamline.append(
    ShadowM2(
        name="m2",
        beam = beam,
        pitch=0.001,
        past_element=beamline[-1]
))

beamline[2].pitch.set(0.005).wait()
beamline[2].read()

In [ ]:
source = ShadowSource(
    name="src",
    beam = beam,
    energy=20,
    delta_energy=1,
    n_rays=100_000
)

RE(scan([source], source.sigmax,  0.033, 0.5, num = 5))

In [ ]:
m2 = beamline[2]
RE(scan([m2], m2.pitch,  0.001, 0.01, num = 5))